# StegaStamp Full Reproduction (Colab)

This notebook runs the full reproduction pipeline:

1. Environment setup
2. Repository setup
3. Encoder/decoder/critic training (all + ablations)
4. Encoded stamp generation for detector training
5. Detector training
6. Synthetic ablation evaluation
7. Full decode-pipeline evaluation

Update the configuration cell before running all cells.

In [ ]:
# Colab setup
!nvidia-smi || true
!python --version
!pip -q install --upgrade pip
!pip -q install torch torchvision pillow lpips bchlib opencv-python numpy

In [ ]:
# OPTIONAL: mount Google Drive
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# Set this to your project folder in Drive (must contain stegastamp/, scripts/, requirements.txt)
REPO_DIR = Path('/content/drive/MyDrive/stegastampOwn')
assert REPO_DIR.exists(), f'Repo dir not found: {REPO_DIR}'

os.chdir(REPO_DIR)
print('Working dir:', Path.cwd())

In [ ]:
# Config
from pathlib import Path

MIRFLICKR_ROOT = Path('/content/drive/MyDrive/datasets/mirflickr_imagefolder')
DIV2K_ROOT = Path('/content/drive/MyDrive/datasets/div2k')
CAPTURED_IMAGES_DIR = Path('/content/drive/MyDrive/datasets/captured_images')
LABELS_JSON = Path('/content/drive/MyDrive/datasets/labels.json')
CAPTURE_METADATA_CSV = Path('/content/drive/MyDrive/datasets/capture_metadata.csv')

RUNS_DIR = Path('runs_colab')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Training controls (set to False to skip parts)
RUN_MAIN_TRAIN = True
RUN_ABLATIONS = True
RUN_GENERATE_STAMPS = True
RUN_DETECTOR_TRAIN = True
RUN_SYNTH_EVAL = True
RUN_PIPELINE_EVAL = False  # enable when captured images + labels are ready

# Hyperparams
ENCDEC_EPOCHS = 30
DETECTOR_EPOCHS = 20
BATCH_SIZE_ENCDEC = 4
BATCH_SIZE_DETECTOR = 2

for p in [MIRFLICKR_ROOT, DIV2K_ROOT]:
    print(p, 'exists:', p.exists())

In [ ]:
# Helpers
import subprocess
import shlex


def run_cmd(cmd: str):
    print('\n>>>', cmd)
    proc = subprocess.run(shlex.split(cmd), check=False)
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with code {proc.returncode}: {cmd}')

In [ ]:
# Train main + ablations
if RUN_MAIN_TRAIN:
    run_cmd(
        f"python scripts/train_encoder_decoder.py "
        f"--data-dir {MIRFLICKR_ROOT} "
        f"--output-dir {RUNS_DIR/'encdec_all'} "
        f"--perturbation-profile all "
        f"--use-bch "
        f"--epochs {ENCDEC_EPOCHS} "
        f"--batch-size {BATCH_SIZE_ENCDEC}"
    )

if RUN_ABLATIONS:
    for profile in ['none', 'pixelwise', 'spatial']:
        run_cmd(
            f"python scripts/train_encoder_decoder.py "
            f"--data-dir {MIRFLICKR_ROOT} "
            f"--output-dir {RUNS_DIR/f'encdec_{profile}'} "
            f"--perturbation-profile {profile} "
            f"--epochs {ENCDEC_EPOCHS} "
            f"--batch-size {BATCH_SIZE_ENCDEC}"
        )

In [ ]:
# Generate encoded stamps for detector training
if RUN_GENERATE_STAMPS:
    import torch
    from torchvision.utils import save_image
    from torch.utils.data import DataLoader
    from stegastamp.data import MirflickrDataset
    from stegastamp.models import StegaStampEncoder

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpt_path = RUNS_DIR / 'encdec_all' / f'checkpoint_epoch_{ENCDEC_EPOCHS-1:03d}.pt'
    assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

    ckpt = torch.load(ckpt_path, map_location=device)
    message_bits = int(ckpt['cfg']['message_bits'])
    image_size = int(ckpt['cfg']['image_size'])

    encoder = StegaStampEncoder(message_bits=message_bits).to(device)
    encoder.load_state_dict(ckpt['encoder'])
    encoder.eval()

    ds = MirflickrDataset(str(MIRFLICKR_ROOT), image_size=image_size)
    dl = DataLoader(ds, batch_size=8, shuffle=True, num_workers=2, drop_last=False)

    stamps_root = RUNS_DIR / 'encoded_stamps'
    stamps_root.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        idx = 0
        for images in dl:
            images = images.to(device)
            msgs = torch.randint(0, 2, (images.size(0), message_bits), device=device).float()
            encoded = encoder(images, msgs).stegastamp
            for b in range(encoded.size(0)):
                save_image(encoded[b], stamps_root / f'stamp_{idx:06d}.png')
                idx += 1
            if idx >= 4000:  # enough synthetic stamps for detector training
                break

    print('Generated stamps in', stamps_root)

In [ ]:
# Train detector
if RUN_DETECTOR_TRAIN:
    run_cmd(
        f"python scripts/train_detector.py "
        f"--backgrounds-dir {DIV2K_ROOT} "
        f"--stamps-dir {RUNS_DIR/'encoded_stamps'} "
        f"--output-dir {RUNS_DIR/'detector'} "
        f"--epochs {DETECTOR_EPOCHS} "
        f"--batch-size {BATCH_SIZE_DETECTOR}"
    )

In [ ]:
# Synthetic ablation evaluation
if RUN_SYNTH_EVAL:
    run_cmd(
        f"python scripts/eval_synthetic_ablation.py "
        f"--checkpoint {RUNS_DIR/'encdec_all'/f'checkpoint_epoch_{ENCDEC_EPOCHS-1:03d}.pt'} "
        f"--data-dir {MIRFLICKR_ROOT} "
        f"--output-dir {RUNS_DIR/'eval_synth_all'}"
    )

In [ ]:
# Full decode-pipeline evaluation (requires captured data + labels)
if RUN_PIPELINE_EVAL:
    assert CAPTURED_IMAGES_DIR.exists(), f'Missing captured images dir: {CAPTURED_IMAGES_DIR}'
    assert LABELS_JSON.exists(), f'Missing labels json: {LABELS_JSON}'
    assert CAPTURE_METADATA_CSV.exists(), f'Missing metadata csv: {CAPTURE_METADATA_CSV}'

    run_cmd(
        f"python scripts/eval_decode_pipeline.py "
        f"--decoder-checkpoint {RUNS_DIR/'encdec_all'/f'checkpoint_epoch_{ENCDEC_EPOCHS-1:03d}.pt'} "
        f"--detector-checkpoint {RUNS_DIR/'detector'/f'detector_epoch_{DETECTOR_EPOCHS-1:03d}.pt'} "
        f"--images-dir {CAPTURED_IMAGES_DIR} "
        f"--labels-json {LABELS_JSON} "
        f"--capture-metadata-csv {CAPTURE_METADATA_CSV} "
        f"--output-dir {RUNS_DIR/'eval_pipeline'} "
        f"--use-bch"
    )

In [ ]:
# Quick outputs check
from pathlib import Path

for p in [
    RUNS_DIR / 'encdec_all',
    RUNS_DIR / 'detector',
    RUNS_DIR / 'eval_synth_all',
    RUNS_DIR / 'eval_pipeline',
]:
    print('\n', p)
    if p.exists():
        for f in sorted(p.glob('*'))[:10]:
            print(' -', f.name)
    else:
        print(' - (missing)')